In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd

In [2]:
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0,1,n_steps)

    series = 0.5 * np.sin(( time - offsets1) * (freq1 * 10 + 10)) #wave 1
    series += 0.5 * np.sin(( time - offsets2) * (freq2 * 20 + 20)) #2
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5) #noise
    return series[..., np.newaxis].astype(np.float32)

In [3]:
n_steps = 50
series = generate_time_series(10000,n_steps+1)

In [4]:
new_n_steps = 1
new_series = generate_time_series(10,new_n_steps+1)

In [5]:
new_series

array([[[-0.2834148 ],
        [ 0.13577259]],

       [[-0.9247585 ],
        [-0.02229544]],

       [[ 0.18806046],
        [-0.00829875]],

       [[ 0.11717911],
        [ 1.0245284 ]],

       [[ 0.740731  ],
        [-0.32420993]],

       [[ 0.04542979],
        [-0.68947715]],

       [[-0.1562227 ],
        [ 0.3058786 ]],

       [[ 0.09496053],
        [-0.86520594]],

       [[-0.5645984 ],
        [ 0.68234235]],

       [[-0.0883794 ],
        [ 0.1189784 ]]], dtype=float32)

In [6]:
X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_valid, y_valid = series[9000:, :n_steps], series[9000:, -1]

In [7]:
y_pred = X_valid[:, -1]
np.mean(keras.losses.MeanSquaredError()(y_valid, y_pred))

np.float32(0.055971254)

In [8]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[50,1]),
    keras.layers.Dense(1)])
model.compile(loss="mse",optimizer="sgd",metrics=["mae"])

C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
history = model.fit(X_train,y_train,epochs=30,validation_data=(X_valid,y_valid))

Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1636 - mae: 0.3089 - val_loss: 0.0674 - val_mae: 0.2101
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 996us/step - loss: 0.0465 - mae: 0.1741 - val_loss: 0.0388 - val_mae: 0.1599
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step - loss: 0.0316 - mae: 0.1427 - val_loss: 0.0294 - val_mae: 0.1386
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 987us/step - loss: 0.0252 - mae: 0.1268 - val_loss: 0.0242 - val_mae: 0.1249
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step - loss: 0.0209 - mae: 0.1152 - val_loss: 0.0203 - val_mae: 0.1146
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 999us/step - loss: 0.0178 - mae: 0.1063 - val_loss: 0.0173 - val_mae: 0.1057
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0154 - mae: 0.0989 - val_loss: 0.0152 - val_mae: 0.0989
Epoch 8/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 975us/step - loss: 0.0135 - mae: 0.0927 - val_loss: 0.0133 - val_mae: 0.0927
Epoch 9/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 9

In [10]:
rnnmodel = keras.models.Sequential([
    keras.layers.SimpleRNN(1, input_shape=[None,1])
])

C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [11]:
rnnmodel.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ (None, 1)                   │               3 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3 (12.00 B)

 Trainable params: 3 (12.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
rnnmodel.compile(loss="mse",optimizer="sgd",metrics=["mae"])

In [13]:
history = rnnmodel.fit(X_train,y_train,epochs=30, validation_data=[X_valid,y_valid])

Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0291 - mae: 0.1414 - val_loss: 0.0275 - val_mae: 0.1378
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0272 - mae: 0.1379 - val_loss: 0.0264 - val_mae: 0.1356
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0267 - mae: 0.1369 - val_loss: 0.0260 - val_mae: 0.1348
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0265 - mae: 0.1366 - val_loss: 0.0258 - val_mae: 0.1344
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0264 - mae: 0.1365 - val_loss: 0.0257 - val_mae: 0.1342
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0264 - mae: 0.1365 - val_loss: 0.0257 - val_mae: 0.1341
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0263 - mae: 0.1364 - val_loss: 0.0257 - val_mae: 0.1340
Epoch 8/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0263 - mae: 0.1364 - val_loss: 0.0256 - val_mae: 0.1340
Epoch 9/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - lo

In [14]:
rnn_fully_model = keras.models.Sequential([
        keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None,1]),
        keras.layers.SimpleRNN(20, return_sequences=True),
        keras.layers.SimpleRNN(1)
])

In [15]:
rnn_fully_model.compile(loss="mse",optimizer="sgd",metrics=["mae"])

In [16]:
history = rnn_fully_model.fit(X_train,y_train, epochs=30, validation_data=[X_valid,y_valid])

Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0789 - mae: 0.2257 - val_loss: 0.0367 - val_mae: 0.1592
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0254 - mae: 0.1296 - val_loss: 0.0198 - val_mae: 0.1140
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0165 - mae: 0.1036 - val_loss: 0.0148 - val_mae: 0.0982
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0133 - mae: 0.0926 - val_loss: 0.0120 - val_mae: 0.0881
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0115 - mae: 0.0858 - val_loss: 0.0110 - val_mae: 0.0839
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0103 - mae: 0.0814 - val_loss: 0.0101 - val_mae: 0.0805
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0096 - mae: 0.0786 - val_loss: 0.0093 - val_mae: 0.0770
Epoch 8/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0090 - mae: 0.0759 - val_loss: 0.0090 - val_mae: 0.0756
Epoch 9/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - l

In [17]:
rnn_fully_model_dense = keras.models.Sequential([
        keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None,1]),
        keras.layers.SimpleRNN(20),
        keras.layers.Dense(1)
])

In [18]:
rnn_fully_model_dense.compile(loss="mse",optimizer="sgd",metrics=["mae"])

In [19]:
history = rnn_fully_model_dense.fit(X_train,y_train, epochs=30, validation_data=[X_valid,y_valid])

Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0447 - mae: 0.1559 - val_loss: 0.0197 - val_mae: 0.1145
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0157 - mae: 0.1009 - val_loss: 0.0130 - val_mae: 0.0923
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0117 - mae: 0.0862 - val_loss: 0.0103 - val_mae: 0.0821
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0097 - mae: 0.0787 - val_loss: 0.0089 - val_mae: 0.0759
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0086 - mae: 0.0741 - val_loss: 0.0079 - val_mae: 0.0719
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0078 - mae: 0.0707 - val_loss: 0.0072 - val_mae: 0.0683
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0071 - mae: 0.0677 - val_loss: 0.0072 - val_mae: 0.0677
Epoch 8/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0067 - mae: 0.0659 - val_loss: 0.0071 - val_mae: 0.0676
Epoch 9/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - lo

In [20]:
series = generate_time_series(1, n_steps + 10)
X_new, Y_new = series[:, :n_steps], series[:, n_steps:]
X = X_new
for step_ahead in range(10):
    y_pred_one = rnn_fully_model_dense.predict(X[:, step_ahead:])[:, np.newaxis, :]
    X = np.concatenate([X,y_pred_one],axis=1)

Y_pred = X[:, n_steps:]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


In [21]:
Y_pred

array([[[-0.07333237],
        [-0.1751323 ],
        [-0.21129802],
        [-0.13480364],
        [ 0.07331179],
        [ 0.34292918],
        [ 0.62677646],
        [ 0.80744594],
        [ 0.76198894],
        [ 0.5557163 ]]], dtype=float32)

In [22]:
series = generate_time_series(10000, n_steps + 10)
X_train, Y_train = series[:7000, :n_steps], series[:7000, -10:, 0]
X_valid, Y_valid = series[7000:9000, :n_steps], series[7000:9000, -10:, 0]
X_test, Y_test = series[9000:, :n_steps], series[9000:, -10:, 0]

In [23]:
model2 = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None,1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(10)
])

In [24]:
model2.compile(loss="mse", optimizer="adam")
model2.fit(X_train, Y_train, epochs=20, validation_data=(X_valid, Y_valid))

Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.1469 - val_loss: 0.0743
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0551 - val_loss: 0.0410
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0386 - val_loss: 0.0403
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0311 - val_loss: 0.0294
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0248 - val_loss: 0.0215
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0220 - val_loss: 0.0196
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0193 - val_loss: 0.0174
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0169 - val_loss: 0.0161
Epoch 9/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0155 - val_loss: 0.0150
Epoch 10/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0150 - val_loss: 0.0137
Epoch 11/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0133 - val_loss: 0.0132
Epoch 12/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step

In [25]:
series_new = generate_time_series(1, n_steps + 10)
X_new, Y_new = series_new[:, :n_steps], series_new[:, n_steps:, 0]
Y_pred = model2.predict(X_new)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step


In [26]:
Y_naive = np.tile(X_valid[:, -1], 10)
print(np.mean((Y_valid - Y_naive) ** 2))
print(model2.evaluate(X_valid, Y_valid))

0.51113164
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0088
0.008771726861596107


In [27]:
Y_naive

array([[ 0.6895678 ,  0.6895678 ,  0.6895678 , ...,  0.6895678 ,
         0.6895678 ,  0.6895678 ],
       [ 0.6557773 ,  0.6557773 ,  0.6557773 , ...,  0.6557773 ,
         0.6557773 ,  0.6557773 ],
       [ 0.20758414,  0.20758414,  0.20758414, ...,  0.20758414,
         0.20758414,  0.20758414],
       ...,
       [-0.8498664 , -0.8498664 , -0.8498664 , ..., -0.8498664 ,
        -0.8498664 , -0.8498664 ],
       [-0.8944415 , -0.8944415 , -0.8944415 , ..., -0.8944415 ,
        -0.8944415 , -0.8944415 ],
       [-0.44830108, -0.44830108, -0.44830108, ..., -0.44830108,
        -0.44830108, -0.44830108]], shape=(2000, 10), dtype=float32)

In [28]:
y = np.empty((10000, n_steps, 10))
for step_ahead in range(1, 11):
    y[:, :, step_ahead-1] = series[:, step_ahead:step_ahead+n_steps, 0]

y_train = y[:7000]
y_valid = y[7000:9000]
y_test = y[9000:]

In [29]:
model3 = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None,1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

In [30]:
def last_time_step_mse(Y_true, Y_pred):
    return tf.reduce_mean(tf.square(Y_true[:, -1] - Y_pred[:, -1]))
optimizer = keras.optimizers.Adam()
model3.compile(loss="mse", optimizer=optimizer, metrics=[last_time_step_mse])

In [31]:
# model3.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid))

In [32]:
class LNSimpleRNNCell(keras.layers.Layer):
    def __init__(self, units, activation="tanh", **kwargs):
        super().__init__(**kwargs)
        self.state_size = units
        self.output_size = units
        self.simple_rnn_cell = keras.layers.SimpleRNNCell(units, activation=None)

    def call(self, inputs, states):
        outputs, new_states = self.simple_rnn_cell(inputs,states)
        norm_outputs = self.activation(self.layer_norm(outputs))
        return norm_outputs, [norm_outputs]

In [33]:
model4 = keras.models.Sequential([
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True, input_shape=[None,1]),
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'ln_simple_rnn_cell', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'ln_simple_rnn_cell_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


In [34]:
lstm_model = model3 = keras.models.Sequential([
    keras.layers.LSTM(20, return_sequences=True, input_shape=[None,1]),
    keras.layers.LSTM(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

In [35]:
lstm_model.compile(loss="mse",optimizer="adam")

In [36]:
model2.fit(X_train, Y_train, epochs=20, validation_data=(X_valid, Y_valid))

Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0097 - val_loss: 0.0116
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0092 - val_loss: 0.0083
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0091 - val_loss: 0.0083
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0090 - val_loss: 0.0082
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0092 - val_loss: 0.0082
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0086 - val_loss: 0.0087
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0083 - val_loss: 0.0091
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0085 - val_loss: 0.0114
Epoch 9/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0080 - val_loss: 0.0073
Epoch 10/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0079 - val_loss: 0.0073
Epoch 11/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0076 - val_loss: 0.0090
Epoch 12/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step

In [2]:
# gru_model = keras.models.Sequential([
#     keras.layers.Conv1D(filters=20,kernel_size=4,strides=2,padding="valid"),
#     keras.layers.GRU(20, return_sequences=True),
#     keras.layers.GRU(20, return_sequences=True),
#     keras.layers.TimeDistributed(keras.layers.Dense(10))
# ])

In [3]:
# for rate in (1,2,4,8) * 2:
#     model.add(keras.layers.Conv1D(filters=20,kernel_size=2, padding="casual", activation="relu", dilation_rate=rate))
# model.add(keras.layers.Conv1D(filters=10,kernel_size=1))
# model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])